# 01 — Exploratory Data Analysis

**Course:** ADS504 — Machine Learning and Deep Learning for Data Science  
**Dataset:** [Bank Marketing (UCI)](https://archive.ics.uci.edu/dataset/222/bank+marketing)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
sns.set_style('whitegrid')


## Load data

Read the full UCI bank marketing file from `data/raw/`.


In [ ]:
DATA_PATH = '../data/raw/bank-full.csv'
df = pd.read_csv(DATA_PATH, sep=';')
print('Shape:', df.shape)
df.head()


## Data overview

Check types, duplicates, and nulls before plotting.


In [ ]:
df.info()
print('Duplicate rows:', df.duplicated().sum())
print('Null cells per column:')
print(df.isnull().sum())


In [ ]:
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
target = 'y'

df[numeric_cols].describe().T.round(1)


No duplicate rows and no NaNs. About 45k contacts with a mix of numeric and categorical features.


## Target variable

Predict whether a client subscribes to a term deposit (`y`).


In [ ]:
df['y_binary'] = (df['y'] == 'yes').astype(int)
base_rate = df['y_binary'].mean() * 100

vc = df['y'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(vc.index, vc.values, color=['steelblue', 'coral'])
ax.set_title('Target distribution (y)')
ax.set_ylabel('count')
for i, v in enumerate(vc.values):
    ax.text(i, v + 400, f'{v/len(df)*100:.1f}%', ha='center')
plt.tight_layout()
plt.show()

print(f'Yes rate: {base_rate:.1f}%')


Only about 12% subscribe. Accuracy will look high if we always predict no, so later we should use recall, F1, and ROC-AUC.


## Missing and coded values

Some fields use `unknown` or `pdays = -1` instead of NaN.


In [ ]:
missing = {c: (df[c] == 'unknown').mean() * 100 for c in categorical_cols if (df[c] == 'unknown').any()}
missing['pdays == -1'] = (df['pdays'] == -1).mean() * 100
miss_s = pd.Series(missing).sort_values(ascending=False)
print(miss_s.round(1))

print('pdays == -1:', (df['pdays'] == -1).sum())
print('poutcome == unknown:', (df['poutcome'] == 'unknown').sum())
print('previous == 0:', (df['previous'] == 0).sum())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
miss_s.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('percent of rows')
ax.set_title('Unknown or sentinel-coded values')
plt.tight_layout()
plt.show()


`poutcome` unknown, `pdays = -1`, and `previous = 0` line up. That means no prior contact, not broken data. Keep unknown as its own level in preprocessing.


## Numeric distributions

Look at shape, skew, and outliers for the continuous fields.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.ravel(), ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']):
    ax.hist(df[col], bins=50, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

print(df[['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']].skew().round(2))


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for ax, col in zip(axes, ['balance', 'duration', 'campaign', 'previous']):
    ax.boxplot(df[col])
    ax.set_title(col)
plt.tight_layout()
plt.show()


Balance, campaign, previous, and duration are right-skewed. Extreme values look real (large balances, many calls), so we should transform them later instead of deleting rows.


## Numeric features by target

Compare numeric distributions for yes vs no.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
plot_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
for ax, col in zip(axes.ravel(), plot_cols):
    for lab, color in [('no', 'steelblue'), ('yes', 'coral')]:
        ax.hist(df.loc[df.y == lab, col], bins=40, alpha=0.55, color=color, label=lab, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
df.groupby('y')[numeric_cols].median().round(1)


Yes clients tend to have longer calls and slightly different prior-contact patterns. Duration stands out the most, which we check next for leakage.


## Correlations

Numeric associations, including the target.


In [ ]:
corr = df[numeric_cols + ['y_binary']].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Numeric correlations')
plt.tight_layout()
plt.show()


Most numeric features are weakly related to `y`. Duration is the exception. `pdays` and `previous` are related to each other.


## Duration and leakage

Duration is call length in seconds. It is only known after the call happens.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lab, color in [('no', 'steelblue'), ('yes', 'coral')]:
    ax[0].hist(df.loc[df.y == lab, 'duration'], bins=60, alpha=0.6, color=color, label=lab, density=True)
ax[0].set_xlim(0, 1500)
ax[0].set_xlabel('duration (sec)')
ax[0].legend()

ax[1].boxplot(
    [df.loc[df.y == 'no', 'duration'], df.loc[df.y == 'yes', 'duration']],
    labels=['no', 'yes']
)
ax[1].set_ylabel('duration (sec)')
plt.tight_layout()
plt.show()

print('Correlation with target:', round(df['duration'].corr(df['y_binary']), 3))


Strong link to the target, but including duration would leak future information. Drop it before modeling.


## Categorical features vs target

Subscription rate by category (red line = overall yes rate).


In [ ]:
def subscription_rate(col, order=None):
    t = df.groupby(col)['y_binary'].agg(['mean', 'count'])
    if order is not None:
        t = t.reindex(order)
    t['rate_pct'] = t['mean'] * 100
    return t.sort_values('rate_pct', ascending=False)

for col in ['job', 'education', 'marital', 'housing', 'loan', 'contact', 'poutcome', 'default']:
    print(f'\n{col}:')
    print(subscription_rate(col).round(1))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

job_rates = subscription_rate('job').sort_values('rate_pct')
axes[0, 0].barh(job_rates.index.astype(str), job_rates['rate_pct'], color='steelblue')
axes[0, 0].axvline(base_rate, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_title('By job')

edu_order = ['primary', 'secondary', 'tertiary', 'unknown']
edu_rates = subscription_rate('education', order=edu_order)
axes[0, 1].bar(edu_rates.index.astype(str), edu_rates['rate_pct'], color='steelblue')
axes[0, 1].axhline(base_rate, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('By education')

contact_rates = subscription_rate('contact')
axes[1, 0].bar(contact_rates.index.astype(str), contact_rates['rate_pct'], color='steelblue')
axes[1, 0].axhline(base_rate, color='red', linestyle='--', linewidth=1)
axes[1, 0].set_title('By contact')

pout_rates = subscription_rate('poutcome')
axes[1, 1].bar(pout_rates.index.astype(str), pout_rates['rate_pct'], color='steelblue')
axes[1, 1].axhline(base_rate, color='red', linestyle='--', linewidth=1)
axes[1, 1].set_title('By previous outcome')

for ax in axes.ravel():
    ax.set_ylabel('rate (%)')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ['housing', 'loan', 'marital']):
    rates = subscription_rate(col)
    ax.bar(rates.index.astype(str), rates['rate_pct'], color='steelblue')
    ax.axhline(base_rate, color='red', linestyle='--', linewidth=1)
    ax.set_title(f'By {col}')
    ax.set_ylabel('rate (%)')
plt.tight_layout()
plt.show()


Biggest categorical signals: previous campaign success, contact type, job, and education. Clients without housing or personal loans convert a bit higher.


## Month: call volume vs conversion

Compare how many calls were made each month with the subscription rate.


In [ ]:
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month_stats = df.groupby('month')['y_binary'].agg(['mean', 'count']).reindex(month_order)
month_stats['rate_pct'] = month_stats['mean'] * 100

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(month_order, month_stats['count'], color='steelblue', alpha=0.8)
ax1.set_ylabel('call volume')
ax2 = ax1.twinx()
ax2.plot(month_order, month_stats['rate_pct'], color='coral', marker='o')
ax2.axhline(base_rate, color='red', linestyle='--', linewidth=1)
ax2.set_ylabel('subscription rate (%)')
ax1.set_title('Calls and conversion by month')
plt.tight_layout()
plt.show()

month_stats.round(1)


May has the most calls but a low conversion rate. Months like March, September, October, and December convert much better with far fewer calls.


## Campaign intensity

How many contacts during this campaign, and how does that relate to the outcome.


In [ ]:
# Cap display for readability; raw campaign goes higher
campaign_cap = df['campaign'].clip(upper=10)
camp = pd.DataFrame({'campaign_capped': campaign_cap, 'y_binary': df['y_binary']})
camp_rates = camp.groupby('campaign_capped')['y_binary'].agg(['mean', 'count'])
camp_rates['rate_pct'] = camp_rates['mean'] * 100

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(camp_rates.index.astype(str), camp_rates['count'], color='steelblue', alpha=0.8)
ax1.set_xlabel('campaign contacts (10 = 10+)')
ax1.set_ylabel('count')
ax2 = ax1.twinx()
ax2.plot(camp_rates.index.astype(str), camp_rates['rate_pct'], color='coral', marker='o')
ax2.axhline(base_rate, color='red', linestyle='--', linewidth=1)
ax2.set_ylabel('subscription rate (%)')
ax1.set_title('Campaign contacts vs conversion')
plt.tight_layout()
plt.show()


Conversion drops as more follow-up calls are made in the same campaign. Extra contacts past the first few look less useful.


## Cross-tabulations

Simple interactions between a few categorical features.


In [ ]:
print('Subscription rate (%) by housing x loan')
print(pd.crosstab(df['housing'], df['loan'], values=df['y_binary'], aggfunc='mean').round(3) * 100)

print('\nSubscription rate (%) by contact x poutcome')
print(pd.crosstab(df['contact'], df['poutcome'], values=df['y_binary'], aggfunc='mean').round(3) * 100)

print('\nSubscription rate (%) by education x marital')
print(pd.crosstab(df['education'], df['marital'], values=df['y_binary'], aggfunc='mean').round(3) * 100)


No housing + no loan does best. Previous success still dominates when combined with contact type.


## Age

Subscription rate across age bands.


In [ ]:
df['age_band'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], labels=['<=30', '31-45', '46-60', '60+'])
age_rates = subscription_rate('age_band')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist([df.loc[df.y == 'no', 'age'], df.loc[df.y == 'yes', 'age']],
             bins=30, label=['no', 'yes'], color=['steelblue', 'coral'], alpha=0.7)
axes[0].set_title('Age distribution by target')
axes[0].legend()

axes[1].bar(age_rates.index.astype(str), age_rates['rate_pct'], color='steelblue')
axes[1].axhline(base_rate, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Subscription rate by age band')
axes[1].set_ylabel('rate (%)')
plt.tight_layout()
plt.show()

age_rates.round(1)


Younger clients and those 60+ convert above average. The middle age bands are closer to the overall rate.


## Balance bands

Rough check of account balance vs subscription.


In [ ]:
df['balance_band'] = pd.qcut(df['balance'], q=4, duplicates='drop')
bal_rates = subscription_rate('balance_band')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(bal_rates)), bal_rates['rate_pct'], color='steelblue')
ax.set_xticks(range(len(bal_rates)))
ax.set_xticklabels([str(x) for x in bal_rates.index], rotation=20, ha='right')
ax.axhline(base_rate, color='red', linestyle='--', linewidth=1)
ax.set_ylabel('rate (%)')
ax.set_title('Subscription rate by balance quartile')
plt.tight_layout()
plt.show()

bal_rates.round(1)


Higher balance quartiles convert a bit more often, but the gap is smaller than for poutcome or month.


## Summary

- 45,211 rows, no duplicates or NaNs. Unknown and `pdays = -1` mostly mean no prior contact.
- Target is imbalanced (~12% yes), so accuracy alone is a weak metric.
- Strong categorical signals: poutcome, contact, job, education, month.
- Campaign intensity and balance matter some; age has a mild U-shape.
- Duration is strongly related to y but is leakage and should be dropped.
- Numeric features are skewed; plan transforms and encoding in preprocessing.
